# ML-07 - Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/8-BitRhyon/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane 4 (provisional): CTR and engagement opportunity scoring for content review.
This notebook checks the two signals the rule leans on, encodes one transparent baseline, writes
the ranked queue to `work/outputs/baseline_action_score.csv`, and reviews the top ten with a
skeptic's eye.

Data: `data/raw/content_refresh_anonymized.csv`, the 30k-row teaching slice the card points to.
The same rule is re-implemented on the warehouse page-month frame in ML-08, where the model is
compared against it.

The lane locks here: CTR opportunity, ranked for a fixed review budget, with no engagement columns.


## 1. Two signal checks

The rule leans on two signals. Each gets a bucket table with `n`, and a one-word verdict.

1. **CTR versus position** - the signal behind the CTR-fix logic. A CTR number only means something
   relative to where the page ranks.
2. **Volume concentration** - the signal behind quick-win scoring. Volume decides whether a fix
   moves enough clicks to matter.

A volume floor is applied to the first table. The data dictionary warns that the `top_3` stratum has
a median of about 53 impressions, where a single click moves CTR by almost two points. Without the
floor the tier comparison is mostly noise.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
REPO_URL = "https://github.com/8-BitRhyon/flyrank-ml-internship"

if "google.colab" in sys.modules and not DATA_REL.exists():
    if not Path("flyrank-ml-internship").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")

while not DATA_REL.exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")

df = pd.read_csv(DATA_REL)
df["tier"] = pd.cut(
    df.avg_position, [0, 3, 10, 20, np.inf],
    labels=["top_3", "page_1", "page_2", "deep"],
).astype(str)

FLOOR = 1500  # impressions over 90 days, about 500 per month

# Signal 1: CTR versus position, with a volume floor.
s1 = df[(df.avg_position > 0) & (df.impressions_90d >= FLOOR)].copy()
g1 = s1.groupby("tier", observed=True)
signal1 = pd.DataFrame({
    "n": g1.size(),
    "median_impressions": g1.impressions_90d.median(),
    "median_page_ctr_pct": g1.ctr.median(),
    "observed_ctr_pct": 100 * g1.clicks_90d.sum() / g1.impressions_90d.sum(),
    "zero_click_share_pct": 100 * g1.clicks_90d.apply(lambda s: (s == 0).mean()),
}).round(2)
signal1 = signal1.reindex(["top_3", "page_1", "page_2", "deep"])
print(f"Signal 1: pages with at least {FLOOR:,} impressions and a real position")
print(signal1.to_string())

# Signal 2: volume concentration.
s2 = df.groupby("impression_tier", observed=True).agg(
    n=("content_id", "size"),
    impressions=("impressions_90d", "sum"),
    clicks=("clicks_90d", "sum"),
)
s2["share_pages_pct"] = 100 * s2.n / s2.n.sum()
s2["share_impressions_pct"] = 100 * s2.impressions / s2.impressions.sum()
s2["share_clicks_pct"] = 100 * s2.clicks / s2.clicks.sum()
s2["observed_ctr_pct"] = 100 * s2.clicks / s2.impressions
signal2 = s2[["n", "share_pages_pct", "share_impressions_pct", "share_clicks_pct", "observed_ctr_pct"]]
signal2 = signal2.reindex(["low", "moderate", "good", "excellent"]).round(2)
print("\nSignal 2: where the traffic sits")
print(signal2.to_string())

big = signal2.loc[["good", "excellent"]]
print(f"\ngood + excellent: {big.n.sum():,} pages ({big.share_pages_pct.sum():.1f}% of pages), "
      f"{big.share_impressions_pct.sum():.1f}% of impressions, {big.share_clicks_pct.sum():.1f}% of clicks")


Signal 1: pages with at least 1,500 impressions and a real position
           n  median_impressions  median_page_ctr_pct  observed_ctr_pct  zero_click_share_pct
tier                                                                                         
top_3    371              6645.0                 0.27              0.49                  3.50
page_1  5548              6734.5                 0.25              0.35                  2.14
page_2  2811              3940.0                 0.19              0.36                  5.91
deep    2891              5051.0                 0.10              0.15                 11.21

Signal 2: where the traffic sits
                     n  share_pages_pct  share_impressions_pct  share_clicks_pct  observed_ctr_pct
impression_tier                                                                                   
low              11248            37.49                   0.52              0.57              0.34
moderate         10469            34.

**Signal 1 verdict: CONFIRMED.** With a volume floor, the median page CTR falls from 0.27% at
`top_3` to 0.10% at `deep`, and observed CTR falls from 0.49% to 0.15%. The gradient is real.

One caveat the table shows: `page_1` and `page_2` are close on observed CTR (0.35% and 0.36%),
even though their medians differ (0.25% and 0.19%). The four-tier split is justified at the top and
bottom but thin in the middle. The rule keeps four tiers to match the data contract, and this
caveat is one reason a model may add value later.

**Signal 2 verdict: CONFIRMED.** The top 27.6% of pages by volume carry 91.4% of impressions and
93.5% of clicks. The bottom 37.5% of pages carry 0.5% of impressions. Weighting a CTR gap by
impressions is therefore not a detail: it is the difference between a queue that can move clicks
and one that cannot.


## 2. My rule and its reason codes

**The rule in plain words.** A page is worth reviewing when it gets enough search impressions to
matter and its CTR is below the median for its position tier. The score is the number of clicks the
page would gain over 90 days if it reached that median.

**Score.**

```
score = max(0, tier_median_ctr - page_ctr) * impressions_90d / 100
```

No fitted weights, no future window, no label-derived columns. The score is an expected click gap,
so it is readable and comparable across pages.

**One reason code and one action label.**

| Field | Value | Meaning |
|---|---|---|
| `reason_code` | `CTR_GAP_VS_TIER` | The page converts below the median for its position tier. |
| `action_label` | `REVIEW_TITLE_META` | A reviewer rewrites the title and meta description, then checks the query mix. |

One code, not many, because a baseline is easier to audit when every row explains itself the same
way.


In [2]:
# Slice: enough traffic for a stable ratio, and a real position value.
queue_df = df[(df.impressions_90d >= FLOOR) & (df.avg_position > 0)].copy()

# Tier medians, computed on the same slice, from current-snapshot values only.
tier_median_ctr = queue_df.groupby("tier", observed=True).ctr.median()

queue_df["expected_click_gap"] = (
    (queue_df.tier.map(tier_median_ctr) - queue_df.ctr).clip(lower=0)
    * queue_df.impressions_90d / 100
)
queue_df["score"] = queue_df.expected_click_gap.round(2)
queue_df["reason_code"] = "CTR_GAP_VS_TIER"
queue_df["action_label"] = "REVIEW_TITLE_META"

queue = (
    queue_df.sort_values(["score", "impressions_90d", "content_id"], ascending=[False, False, True])
    .reset_index(drop=True)
)
queue["rank"] = queue.index + 1

print(f"pages in the queue: {len(queue):,}")
print(f"tier medians used: {tier_median_ctr.round(3).to_dict()}")
print(f"total expected click gap: {queue.score.sum():,.0f} clicks per 90 days")
print(f"top 140 captures: {queue.head(140).score.sum():,.0f} clicks "
      f"({100 * queue.head(140).score.sum() / queue.score.sum():.1f}% of the total gap)")
print(queue[["rank", "content_id", "tier", "impressions_90d", "clicks_90d", "ctr",
             "avg_position", "score", "reason_code", "action_label"]].head(10).to_string(index=False))


pages in the queue: 11,621
tier medians used: {'deep': 0.1, 'page_1': 0.25, 'page_2': 0.19, 'top_3': 0.27}
total expected click gap: 67,901 clicks per 90 days
top 140 captures: 21,300 clicks (31.4% of the total gap)
 rank           content_id   tier  impressions_90d  clicks_90d  ctr  avg_position  score     reason_code      action_label
    1 content_8451fc6f034d  top_3           272144          75 0.03           2.3 653.15 CTR_GAP_VS_TIER REVIEW_TITLE_META
    2 content_8c19996aa890  top_3           509252         785 0.15           2.5 611.10 CTR_GAP_VS_TIER REVIEW_TITLE_META
    3 content_36ff89c8214e page_1           295097         154 0.05           7.3 590.19 CTR_GAP_VS_TIER REVIEW_TITLE_META
    4 content_5fe46e04994d page_1           517715         741 0.14           4.2 569.49 CTR_GAP_VS_TIER REVIEW_TITLE_META
    5 content_c8e9d6ab9013 page_1           208678           0 0.00           9.7 521.70 CTR_GAP_VS_TIER REVIEW_TITLE_META
    6 content_c84a0ab98e90 page_1           22

## 3. Build the ranked queue (writes the CSV)

The queue is written from this notebook to `work/outputs/baseline_action_score.csv`. That file is
gitignored by design: CI blocks data files, and the notebook regenerates it on every run.

The columns are the ones a reviewer needs: identity, the inputs behind the score, the score itself,
the reason code, and the action label.


In [3]:
out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "baseline_action_score.csv"

COLS = ["rank", "content_id", "client_id", "tier", "impressions_90d", "clicks_90d",
        "ctr", "avg_position", "score", "reason_code", "action_label"]
queue[COLS].to_csv(csv_path, index=False)

metrics = {
    "signals": {"ctr_vs_position": "CONFIRMED", "volume_concentration": "CONFIRMED"},
    "rule": {
        "score": "max(0, tier_median_ctr - ctr) * impressions_90d / 100",
        "reason_code": "CTR_GAP_VS_TIER",
        "action_label": "REVIEW_TITLE_META",
    },
    "queue": {
        "rows": int(len(queue)),
        "impressions_floor": FLOOR,
        "total_expected_click_gap": round(float(queue.score.sum()), 2),
        "top140_gap": round(float(queue.head(140).score.sum()), 2),
        "top140_share_of_gap_pct": round(100 * float(queue.head(140).score.sum()) / float(queue.score.sum()), 2),
    },
}
json_path = out_dir / "ml07_baseline_metrics.json"
json_path.write_text(json.dumps(metrics, indent=2) + "\n")

print(f"wrote {csv_path} ({len(queue):,} rows)")
print(f"wrote {json_path}")
print("csv is gitignored; the metrics json is the committed receipt")


wrote work/outputs/baseline_action_score.csv (11,621 rows)
wrote work/outputs/ml07_baseline_metrics.json
csv is gitignored; the metrics json is the committed receipt


## 4. Top-10 review

Each line gives the action, why the page is in the queue, and what would make the call wrong.
All ten are high-volume pages, because the score weights the CTR gap by impressions.

**1. `content_8451fc6f034d`** - `top_3`, 272,144 impressions, 75 clicks, 0.03% CTR, position 2.3.
Action: `REVIEW_TITLE_META`. Why: at position 2.3 the tier median is 0.27%, so the gap is worth
about 653 clicks per 90 days. Wrong if: the query mix is mostly zero-click informational queries
where the SERP answers without a visit. A position 2.3 page with almost no clicks is unusual enough
to check the query report first.

**2. `content_8c19996aa890`** - `top_3`, 509,252 impressions, 785 clicks, 0.15% CTR, position 2.5.
Action: `REVIEW_TITLE_META`. Why: the highest-volume page in the queue, at 0.12 points below its
tier median. Wrong if: 0.15% is normal for its head terms because a featured snippet or AI answer
absorbs the clicks. Check the query mix before rewriting.

**3. `content_36ff89c8214e`** - `page_1`, 295,097 impressions, 154 clicks, 0.05% CTR, position 7.3.
Action: `REVIEW_TITLE_META`. Why: a 0.05% CTR against a page_1 median of 0.25%, on a page that is
already on page one. Wrong if: the impressions come from one broad query the page does not really
answer. A snippet rewrite cannot fix an intent mismatch.

**4. `content_5fe46e04994d`** - `page_1`, 517,715 impressions, 741 clicks, 0.14% CTR, position 4.2.
Action: `REVIEW_TITLE_META`. Why: the largest impression base in the queue, with CTR below its tier
median. Wrong if: the page ranks across a wide long tail where 0.14% is normal, or a site-wide
change depressed CTR in the same window.

**5. `content_c8e9d6ab9013`** - `page_1`, 208,678 impressions, 0 clicks, 0.00% CTR, position 9.7.
Action: check tracking before `REVIEW_TITLE_META`. Why: 208k impressions and zero clicks is the
strongest gap in the queue. Wrong if: the zero is a measurement or attribution failure, or the page
ranks for queries it cannot answer. This is the weakest pick in the ten.

**6. `content_c84a0ab98e90`** - `page_1`, 223,271 impressions, 70 clicks, 0.03% CTR, position 7.8.
Action: `REVIEW_TITLE_META`. Why: high volume with a CTR far below the page_1 median. Wrong if: at
95 days the page is still settling into its rankings, and CTR may move without an edit.

**7. `content_453722754fea`** - `page_1`, 140,079 impressions, 16 clicks, 0.01% CTR, position 7.6.
Action: `REVIEW_TITLE_META`. Why: a 0.01% CTR on 140k impressions. Wrong if: the page duplicates
another URL and should be consolidated or canonicalised instead of rewritten.

**8. `content_4a6607efcb46`** - `top_3`, 128,068 impressions, 17 clicks, 0.01% CTR, position 2.2.
Action: `REVIEW_TITLE_META`. Why: position 2.2 with 0.01% CTR is almost impossible for a normal
query. Wrong if: the page serves zero-click queries, or the URL is a redirect or AMP variant that
should not be reviewed as content.

**9. `content_73c54f78c06a`** - `page_1`, 213,963 impressions, 211 clicks, 0.10% CTR, position 4.7.
Action: `REVIEW_TITLE_META`. Why: high volume and CTR below the page_1 median. Wrong if: the intent
mix includes queries where 0.10% is the normal rate, which the query report would show.

**10. `content_91652435f57a`** - `page_1`, 159,590 impressions, 100 clicks, 0.06% CTR, position 7.8.
Action: `REVIEW_TITLE_META`. Why: commercial intent usually clicks more, so 0.06% is a strong
mismatch signal. Wrong if: the page ranks for comparison queries where users leave for another site,
or the commercial intent label is wrong.

**Common thread.** Every pick assumes the CTR gap is recoverable with a snippet edit. The review
above names the three ways that assumption fails: the queries have no click intent, the measurement
is broken, or the page has a structural problem a rewrite cannot fix.


## 5. Weak picks + leakage check

Three weak picks, taken from the top twenty rather than the top ten, because that is where the rule
starts to show its seams.

- **Zero-click pages.** Rank 5 has 208,678 impressions and zero clicks. The rule reads that as a
  0.00% CTR gap, the largest in the queue. A tracking failure produces exactly the same signature.
  Review the page's query report and analytics before spending a slot on it.
- **Top-3 pages with near-zero CTR.** Ranks 1 and 8 sit at positions 2.2 and 2.3 with 0.01% to 0.03%
  CTR. That pattern is more often zero-click queries or a redirect than a bad title. The rule cannot
  tell those apart.
- **Tier-boundary sensitivity.** A page at position 9.9 and a page at 10.1 land in different tiers
  with medians of 0.25% and 0.19%. A small ranking change can move a page between buckets and change
  its score without any real change in opportunity.

The leakage check: the rule uses only current-snapshot columns. `trend_direction`, `trend_pct`, and
`is_declining_label` are excluded, so no label-derived or future-window input reaches the score.


In [4]:
forbidden = ["trend_direction", "trend_pct", "is_declining_label"]
rule_inputs = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "tier"]
assert not set(forbidden) & set(rule_inputs), "label-derived input in the rule"
assert not set(forbidden) & set(COLS), "label-derived column in the queue"

zero_click = queue[(queue.clicks_90d == 0) & (queue.impressions_90d >= 50000)]
boundary = queue[queue.avg_position.between(9.5, 10.5)]

print("rule inputs:", rule_inputs)
print("label-derived inputs:", 0)
print(f"\nzero-click pages above 50k impressions: {len(zero_click)}")
print(zero_click[["rank", "content_id", "impressions_90d", "avg_position", "score"]].head(5).to_string(index=False))
print(f"\npages within 0.5 of the page_1/page_2 boundary: {len(boundary)}")
print(f"queue rows: {len(queue):,} | csv exists: {csv_path.exists()} | metrics exists: {json_path.exists()}")


rule inputs: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'tier']
label-derived inputs: 0

zero-click pages above 50k impressions: 1
 rank           content_id  impressions_90d  avg_position  score
    5 content_c8e9d6ab9013           208678           9.7  521.7

pages within 0.5 of the page_1/page_2 boundary: 541
queue rows: 11,621 | csv exists: True | metrics exists: True


## Self-check

- [x] Two signals checked first, each with a bucket table and `n` printed
- [x] Both signals are flag-linked: CTR versus position, and volume for quick-win scoring
- [x] One-word verdict per signal: CONFIRMED for both, with the middle-tier caveat stated
- [x] One rule with a score, one reason code, and one action label
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Top ten reviewed line by line with action, why, and what would make it wrong
- [x] Weak picks named, including the zero-click case and tier-boundary sensitivity
- [x] No future-window or label-derived inputs. The notebook asserts this in code
- [x] The notebook ran top to bottom with no errors (executed before commit)
- [x] No client names, URLs, private queries, or tokens anywhere
- [x] Committed under `work/notebooks/`
